# Expansão de mercado — onde um ISP deve investir primeiro

Narrativa da análise que `run_analysis.py` executa. **Este notebook não
reimplementa nada**: importa `sidra.py`, a mesma fonte que o script e que o
projeto de Power BI usam.

Isso é deliberado. A versão anterior deste notebook tinha a própria cópia do
parsing do SIDRA e da montagem do dataset — e quando o script foi corrigido, o
notebook ficou para trás, ainda construindo um recorte urbano × rural por UF que
o IBGE não publica. Duas implementações da mesma análise divergem; é só questão
de quando.


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from sidra import carregar

ANO_REF = 2023
linhas = carregar()
print(f'{len(linhas)} linhas observadas (nível × localidade × situação × ano)')


## 1. O denominador do problema

Antes de qualquer ranking: quanto ainda falta, e onde a série está.


In [ ]:
br = pd.DataFrame([l for l in linhas if l['nivel'] == 'N1' and l['situacao'] == 'Total'])
serie = br.set_index('ano')['pct'].sort_index()
sem_2023 = (br[br.ano == ANO_REF].total - br[br.ano == ANO_REF].com_internet).iloc[0]

print(serie.to_string())
print(f'\n{ANO_REF}: {serie[ANO_REF]}% dos domicílios · faltam {sem_2023:,.0f} mil')
print('IBGE publica 92,5% para 2023 — a diferença é arredondamento das duas tabelas.')
print('2020 não aparece: a PNAD não coletou o módulo de TIC naquele ano.')


O ganho anual caiu de **+5,5pp (2017)** para **+1,3pp (2025)**. A fase de expansão
orgânica acabou — o que resta é o domicílio que o mercado deixou por último.


## 2. H1 — a brecha regional ainda decide investimento?

Hipótese declarada antes de olhar o resultado: *há gap relevante entre
Norte+Nordeste e Sul+Sudeste*.


In [ ]:
UF_REG = {'11':'Norte','12':'Norte','13':'Norte','14':'Norte','15':'Norte',
          '16':'Norte','17':'Norte','21':'Nordeste','22':'Nordeste',
          '23':'Nordeste','24':'Nordeste','25':'Nordeste','26':'Nordeste',
          '27':'Nordeste','28':'Nordeste','29':'Nordeste','31':'Sudeste',
          '32':'Sudeste','33':'Sudeste','35':'Sudeste','41':'Sul','42':'Sul',
          '43':'Sul','50':'Centro-Oeste','51':'Centro-Oeste','52':'Centro-Oeste',
          '53':'Centro-Oeste'}

uf = pd.DataFrame([l for l in linhas
                   if l['nivel'] == 'N3' and l['situacao'] == 'Total'
                   and l['ano'] == ANO_REF])
uf['regiao'] = uf['codigo_ibge'].map(UF_REG)
uf['sigla']  = uf['codigo_ibge'].map({'11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO','21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL','28':'SE','29':'BA','31':'MG','32':'ES','33':'RJ','35':'SP','41':'PR','42':'SC','43':'RS','50':'MS','51':'MT','52':'GO','53':'DF'})
uf['sem_k']  = (uf['total'] - uf['com_internet']).round(0)

nne = uf[uf.regiao.isin(['Norte', 'Nordeste'])]
sse = uf[uf.regiao.isin(['Sul', 'Sudeste'])]
gap = (sse.com_internet.sum() / sse.total.sum()
       - nne.com_internet.sum() / nne.total.sum()) * 100
print(f'Gap Norte+NE vs Sul+SE: {gap:.1f}pp -> ' +
      ('CONFIRMADA' if gap > 8 else 'REFUTADA'))


**Refutada — 3,5pp.** É o resultado mais útil da análise.

Uma versão anterior deste projeto dava H1 como confirmada com 11,1pp, mas rodava
sobre percentuais escritos à mão que não vinham do PNAD. Com dado observado o gap
regional praticamente sumiu: quem ainda planeja expansão por mapa de região está
resolvendo um problema que encolheu.


## 3. Então onde está a exclusão? No campo.

O recorte existe em Brasil e Grandes Regiões — o IBGE suprime por UF, porque a
amostra da PNAD não sustenta o cruzamento. O grão é o que o dado tem, não o que
ficaria mais bonito.


In [ ]:
REG = {'1':'Norte','2':'Nordeste','3':'Sudeste','4':'Sul','5':'Centro-Oeste'}
sit = {}
for l in linhas:
    if l['ano'] != ANO_REF or l['nivel'] not in ('N1', 'N2'):
        continue
    nome = 'Brasil' if l['nivel'] == 'N1' else REG.get(l['codigo_ibge'])
    if nome:
        sit.setdefault(nome, {})[l['situacao']] = l['pct']

gap_df = (pd.DataFrame(sit).T[['Urbana', 'Rural']]
            .assign(gap=lambda d: (d.Urbana - d.Rural).round(1))
            .sort_values('gap', ascending=False))
gap_df


**O Norte urbano tem 95,2% de acesso — acima da média nacional de 92,6%.**
O rural tem 70,4%, o pior do país. Gap de 24,8pp, três vezes e meia o do
Centro-Oeste.

Para um ISP isso troca a natureza do investimento: no Norte, a cidade é mercado
disputado, não praça a cobrir. A oportunidade de cobertura é rural — outro custo
por assinante, outro prazo de retorno, outra tecnologia de última milha. Tratar
a região como praça única leva à decisão errada nos dois sentidos.


## 4. H2 — taxa e volume apontam para o mesmo lugar?


In [ ]:
from scipy.stats import spearmanr

rho = spearmanr(uf['pct'], -uf['sem_k']).correlation
pior_taxa = set(uf.nsmallest(5, 'pct').sigla)
maior_vol = set(uf.nlargest(5, 'sem_k').sigla)
print(f'rho = {rho:.2f} · {len(pior_taxa & maior_vol)} de 5 coincidem')

pd.concat([
    uf.nsmallest(5, 'pct')[['sigla', 'pct']]
      .reset_index(drop=True).add_prefix('pior_taxa_'),
    uf.nlargest(5, 'sem_k')[['sigla', 'sem_k', 'pct']]
      .reset_index(drop=True).add_prefix('maior_vol_'),
], axis=1)


**Confirmada.** São Paulo tem a *melhor* taxa do país e o *maior* número absoluto
de domicílios desconectados: 852 mil. O Acre tem a pior taxa e 43 mil.

Quem prioriza pelo percentual vai para o Acre; quem prioriza por mercado
endereçável vai para São Paulo. São decisões de capex opostas, tiradas do mesmo
dado — e é por isso que o score de oportunidade combina os dois de forma
declarada, em vez de escolher um em silêncio.

---

O ranking completo e a estratégia por estado saem de `market_scoring.py`.
As seis figuras interativas saem de `run_analysis.py`.
